# Create Evaluation Dataset for Whisper Fine-tuning

This notebook is responsible for creating a holdout evaluation dataset to assess the performance of our fine-tuned Whisper model. We'll create this dataset by:

1. Sampling sentences that were not used in training
2. Preparing metadata for audio recordings
3. Setting up the directory structure for evaluation

In [20]:
from whisper_tune.utils import get_project_root
import random
from typing import List, Dict
import os
import json

## Import Required Libraries

We'll import necessary Python libraries for file handling, random sampling, and type hints.

In [5]:
training_sample = get_project_root() / "data/sampled_corpus.txt"
all_sample = get_project_root() / "data/final_filtered_newest.txt"

with open(training_sample, "r") as f:
    training_lines = f.readlines()
with open(all_sample, "r") as f:
    all_lines = f.readlines()

print(f"Training sample size: {len(training_lines)}")
print(f"All sample size: {len(all_lines)}")

Training sample size: 32000
All sample size: 2153530


## Load Training and Full Datasets

We'll load two datasets:
1. The training sample that was used for fine-tuning
2. The complete filtered dataset

This allows us to ensure our evaluation set doesn't overlap with the training data.

In [16]:
random.seed(42)
sampled_lines = []
while len(sampled_lines) < 100:
    if not all_lines:
        break
    idx = random.randrange(len(all_lines))
    line = all_lines.pop(idx)
    if line not in training_lines:
        sampled_lines.append(line)
print(f"Sampled lines size: {len(sampled_lines)}")

Sampled lines size: 100


## Sample Holdout Sentences

We'll randomly sample 100 sentences from the full dataset, ensuring that none of them were used in training. This creates an independent test set for evaluation.

Note: We use a fixed random seed (42) for reproducibility.

In [17]:
sampled_lines[0:10]

['ин коро имсол онхо худашон ичро менамоянд \n',
 'ана ҳамин тоҷикписар дар ҷӯшу хурӯши ҷанг коре кардааст бебаҳо раҳмон сафар ман истода ба чашмонам дида медӯзад \n',
 'бекорӣ дар ҳавзаи евро ба ҳадди рекордӣ расид бекорӣ дар кишвари ҳавзаи евро ба ҳадде расидааст ки то кунун мушоҳида нашуда буд \n',
 'аз пашша фил сози хуб нест \n',
 'чунин баъдтар дар нимаи дувв асри хх ва ибтидои асри хх дар зери таъсири матлуби илм ва фарҳанги ҷаҳонӣ дар шаҳрҳои тошканду самарқанд бухоро пайдо шуданд \n',
 'мехостам фаҳмам ки вақте ягон чизи медузданд ин ягон хубӣ дорад ё на \n',
 'зеро созмони растохез ҳамеша барои ҳадафҳои миллӣ демократӣ ва озодихоҳӣ мубориза мебурд на барои мансабу ҷоҳу ҷалол \n',
 'дар рафти сафари мазкур раиси суди тоикистон мамудов ма аз суди конститутсионии замини берлин низ дидан намуда бо раис ва судяои он мулоот намуд \n',
 'хеле душвор астгуфт ин узви ҳизби коммунист \n',
 'ба қавли мардум ин ҳамон обро лой кардану моҳӣ гирифтан аст ки дар ин ҷо манфиати танҳо шахс дар

In [22]:
def create_metadata(
    sentences: List[str],
    output_json: str,
    output_audio_dir: str,
    filename_prefix: str = "human_recording",
):
    os.makedirs(output_audio_dir, exist_ok=True)
    metadata = []

    for idx, text in enumerate(sentences):
        audio_path = os.path.join(output_audio_dir, f"{filename_prefix}_{idx:06d}.wav")
        metadata.append({"audio_path": audio_path, "text": text, "duration": None})

    with open(output_json, "w", encoding="utf-8") as f:
        for entry in metadata:
            f.write(json.dumps(entry, ensure_ascii=False) + "\n")

    print(f"Created metadata for {len(sentences)} sentences at {output_json}")
    return metadata

## Create Metadata Generation Function

This function creates the necessary metadata structure for our evaluation dataset:
- Generates unique filenames for audio recordings
- Creates a JSON metadata file with audio paths and transcriptions
- Sets up the directory structure for audio files

In [23]:
metadata = create_metadata(
    sampled_lines,
    output_json=get_project_root() / "data/holdout/metadata.json",
    output_audio_dir=get_project_root() / "data/holdout/audio",
    filename_prefix="human_recording",
)

Created metadata for 100 sentences at /Users/djuraev/0_git/whisper_tune/data/holdout/metadata.json


## Generate Evaluation Dataset Metadata

Finally, we'll create the metadata for our holdout set. This will:
1. Set up the directory structure in `data/holdout`
2. Create a metadata file mapping audio files to transcriptions
3. Prepare file paths for recording audio samples